In [16]:
import pandas as pd, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Conv1D, MaxPooling1D, Dense, Dropout, Flatten
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

PATH        = "data_without_straw.xlsx"
# PATH        = "data_with_straw.xlsx"
df = (pd.read_excel(PATH)
        .sort_values(["Participant_ID", "Time"])
        .reset_index(drop=True))

# clf = tf.keras.models.load_model("Best_gesture_recognition_model.h5")

In [17]:
print("Columns in file:\n", df.columns.tolist(), "\n")

Columns in file:
 ['Time', 'Zone_0', 'Zone_1', 'Zone_2', 'Zone_3', 'Zone_4', 'Zone_5', 'Zone_6', 'Zone_7', 'Zone_8', 'Zone_9', 'Zone_10', 'Zone_11', 'Zone_12', 'Zone_13', 'Zone_14', 'Zone_15', 'Zone_16', 'Zone_17', 'Zone_18', 'Zone_19', 'Zone_20', 'Zone_21', 'Zone_22', 'Zone_23', 'Zone_24', 'Zone_25', 'Zone_26', 'Zone_27', 'Zone_28', 'Zone_29', 'Zone_30', 'Zone_31', 'Zone_32', 'Zone_33', 'Zone_34', 'Zone_35', 'Zone_36', 'Zone_37', 'Zone_38', 'Zone_39', 'Zone_40', 'Zone_41', 'Zone_42', 'Zone_43', 'Zone_44', 'Zone_45', 'Zone_46', 'Zone_47', 'Zone_48', 'Zone_49', 'Zone_50', 'Zone_51', 'Zone_52', 'Zone_53', 'Zone_54', 'Zone_55', 'Zone_56', 'Zone_57', 'Zone_58', 'Zone_59', 'Zone_60', 'Zone_61', 'Zone_62', 'Zone_63', 'Label', 'Volume', 'Participant_ID', 'Actual_Volume', 'Straw', 'Gender', 'Container_Weight', 'drink', 'temp'] 



In [18]:
print("---- Quick stats ------------------------------------------------")
print("Rows                    :", len(df))
print("Participants            :", df['Participant_ID'].nunique())
print("Frames flagged drinking :", (df["Label"]=="Drinking").sum())
print("Frames flagged Not-drinking :", (df["Label"]=="Not_Drinking").sum())
print("Any NaN in ActualVolume :", df['Actual_Volume'].isna().any())
print(df[['Actual_Volume']].describe())

---- Quick stats ------------------------------------------------
Rows                    : 36212
Participants            : 26
Frames flagged drinking : 10081
Frames flagged Not-drinking : 26131
Any NaN in ActualVolume : False
       Actual_Volume
count   36212.000000
mean      154.575279
std        87.472143
min         0.000000
25%       100.000000
50%       120.000000
75%       200.000000
max       320.000000


In [19]:
df 

,Time,Zone_0,Zone_1,Zone_2,Zone_3,Zone_4,Zone_5,Zone_6,Zone_7,Zone_8,...,Zone_63,Label,Volume,Participant_ID,Actual_Volume,Straw,Gender,Container_Weight,drink,temp
0,1750614851000,290,309,2049,2033,320,306,2065,1496,347,...,266,Not_Drinking,300,1,300,N,Male,12.7,water,n
1,1750614851200,290,309,2043,2021,308,322,2065,1609,3731,...,1815,Not_Drinking,300,1,300,N,Male,12.7,water,n
2,1750614851400,290,309,2049,2033,320,306,2065,1505,3731,...,1815,Not_Drinking,300,1,300,N,Male,12.7,water,n
3,1750614852000,290,309,2043,2037,308,314,2170,1531,3731,...,1852,Not_Drinking,300,1,300,N,Male,12.7,water,n
4,1750614852200,290,309,2038,2037,291,314,2196,1531,3731,...,1900,Not_Drinking,300,1,300,N,Male,12.7,water,n
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36207,1750885897800,1718,2310,2415,2597,2625,2644,2671,2667,1627,...,993,Not_Drinking,0,26,0,N,Male,7.7,water,n
36208,1750885898000,1833,2310,2415,2585,2607,2640,2670,2667,2139,...,915,Not_Drinking,0,26,0,N,Male,7.7,water,n
36209,1750885898200,2152,2316,2444,2585,2607,2640,2647,2679,2158,...,909,Not_Drinking,0,26,0,N,Male,7.7,water,n
36210,1750885898400,2152,2316,2444,2585,2607,2640,2647,2679,2158,...,909,Not_Drinking,0,26,0,N,Male,7.7,water,n


In [20]:

id_col      = "Participant_ID"
time_col    = "Time"
flag_col    = "Label"           # 'Drinking' / 'Not_Drinking'
vol_col     = "Actual_Volume"

# ────────────────────────────────────────────────────────────────
# 1. LOAD  +  MAP LABEL -> 0/1
# ────────────────────────────────────────────────────────────────


df[flag_col] = df["Label"].map({"Drinking": 1, "Not_Drinking": 0})


# ────────────────────────────────────────────────────────────────
# 2. FIND SIP BOUNDARIES
# ────────────────────────────────────────────────────────────────
df["prev"] = df.groupby(id_col)["Label"].shift(fill_value=0)
df["next"] = df.groupby(id_col)["Label"].shift(-1, fill_value=0)

df["sip_start"] = (df["prev"] == 0) & (df["Label"] == 1)        # first Drinking frame
df["sip_end"]   = (df["Label"] == 1) & (df["next"] == 0)        # last Drinking frame
df["sip_id"]    = df.groupby(id_col)["sip_start"].cumsum()       # 1,2,3,…

# rows where sip_start / sip_end are True
starts = df.loc[df["sip_start"]]
ends   = df.loc[df["sip_end"]]

# ────────────────────────────────────────────────────────────────
# 3.  PICK VOLUME BEFORE-START  &  AFTER-END
#     (guard against boundary rows)
# ────────────────────────────────────────────────────────────────
vol_before = (df.loc[starts.index - 1, vol_col]          # prev row
                .reset_index(drop=True)
                .rename("vol_before"))
vol_after  = (df.loc[ends.index + 1, vol_col]            # next row
                .reset_index(drop=True)
                .rename("vol_after"))

sip_delta = (pd.concat([starts[[id_col, "sip_id"]].reset_index(drop=True),
                        vol_before,
                        vol_after], axis=1)
               .dropna(subset=["vol_before", "vol_after"]))   # drop edges

sip_delta["dV"] = sip_delta["vol_before"] - sip_delta["vol_after"]
print("Non-zero ΔV rows:", (sip_delta["dV"] != 0).sum(), "/", len(sip_delta))
print(sip_delta["dV"].describe())

assert (sip_delta["dV"] != 0).any(), "ΔV still zero – check volume logging!"

# attach dV to every Drinking frame so we can build sequences
df = df.merge(sip_delta[[id_col, "sip_id", "dV"]],
              on=[id_col, "sip_id"], how="left")


Non-zero ΔV rows: 131 / 132
count    132.000000
mean      56.818182
std       42.432131
min     -300.000000
25%       40.000000
50%       60.000000
75%       80.000000
max      100.000000
Name: dV, dtype: float64


In [21]:
df

,Time,Zone_0,Zone_1,Zone_2,Zone_3,Zone_4,Zone_5,Zone_6,Zone_7,Zone_8,...,Gender,Container_Weight,drink,temp,prev,next,sip_start,sip_end,sip_id,dV
0,1750614851000,290,309,2049,2033,320,306,2065,1496,347,...,Male,12.7,water,n,0,0,False,False,0,NaN
1,1750614851200,290,309,2043,2021,308,322,2065,1609,3731,...,Male,12.7,water,n,0,0,False,False,0,NaN
2,1750614851400,290,309,2049,2033,320,306,2065,1505,3731,...,Male,12.7,water,n,0,0,False,False,0,NaN
3,1750614852000,290,309,2043,2037,308,314,2170,1531,3731,...,Male,12.7,water,n,0,0,False,False,0,NaN
4,1750614852200,290,309,2038,2037,291,314,2196,1531,3731,...,Male,12.7,water,n,0,0,False,False,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36207,1750885897800,1718,2310,2415,2597,2625,2644,2671,2667,1627,...,Male,7.7,water,n,0,0,False,False,5,100.0
36208,1750885898000,1833,2310,2415,2585,2607,2640,2670,2667,2139,...,Male,7.7,water,n,0,0,False,False,5,100.0
36209,1750885898200,2152,2316,2444,2585,2607,2640,2647,2679,2158,...,Male,7.7,water,n,0,0,False,False,5,100.0
36210,1750885898400,2152,2316,2444,2585,2607,2640,2647,2679,2158,...,Male,7.7,water,n,0,0,False,False,5,100.0


In [22]:
test=df.head(500) 
test.to_excel('sipadded.xlsx')

In [23]:
# M=pd.DataFrame(X)
# M=M.head(100) 
# M.to_exel('x.xlsx')

In [24]:
import pandas as pd
import numpy as np

# Display the initial structure of the dataset
# print("Initial DataFrame:")
# print(df.head())

# 1. Replace dV values for non-drinking parts and non-null NaN values
df['dV'] = df.apply(lambda row: 0 if row['Label'] == 0 or pd.isna(row['dV']) else row['dV'], axis=1)
# Count the number of rows for each Participant_ID and sip_id where Label == 1
# sip_time_counts = df[df['Label'] == 1].groupby(['Participant_ID', 'sip_id']).size().reset_index(name='SIP_TIME')

# # Step 3: Merge sip_time_counts back with the original DataFrame to assign SIP_TIME
# df = df.merge(sip_time_counts, on=['Participant_ID', 'sip_id'], how='left')

# # Step 4: Fill NaN values in SIP_TIME with 0 (for non-drinking sips)
# df['SIP_TIME'] = df['SIP_TIME'].fillna(0).astype(int)
# df.loc[df['Label'] == 0, 'SIP_TIME'] = 0
# df['SIP_TIME'] = df.apply(lambda row: 1 if row['Label'] == 1 else 0, axis=1)

# Step 3: Create a total count column for each Participant_ID and sip_id
df['TOTAL_SIP_TIME'] = df.groupby(['Participant_ID', 'sip_id'])['Label'].transform('sum')

# Step 4: Set values to 0 for non-drinking entries
df.loc[df['Label'] == 0, 'TOTAL_SIP_TIME'] = 0
# Display the number of rows and the updated DataFrame
# print("\nUpdated DataFrame with dV replaced:")
# print(df[['Participant_ID', 'sip_id', 'Label', 'dV']].head(20))
test=df.head(500) 
test.to_excel('sipadded.xlsx')
df.to_excel('dv0_withoutstraw_sipCAdded.xlsx')
# df.to_excel('dv0_withstraw_sipCAdded.xlsx')

print(df)

                Time  Zone_0  Zone_1  Zone_2  Zone_3  Zone_4  Zone_5  Zone_6  \
0      1750614851000     290     309    2049    2033     320     306    2065   
1      1750614851200     290     309    2043    2021     308     322    2065   
2      1750614851400     290     309    2049    2033     320     306    2065   
3      1750614852000     290     309    2043    2037     308     314    2170   
4      1750614852200     290     309    2038    2037     291     314    2196   
...              ...     ...     ...     ...     ...     ...     ...     ...   
36207  1750885897800    1718    2310    2415    2597    2625    2644    2671   
36208  1750885898000    1833    2310    2415    2585    2607    2640    2670   
36209  1750885898200    2152    2316    2444    2585    2607    2640    2647   
36210  1750885898400    2152    2316    2444    2585    2607    2640    2647   
36211  1750885898600    2152    2316    2444    2585    2607    2640    2647   

       Zone_7  Zone_8  ...  Container_W

In [25]:
from sklearn.model_selection import train_test_split

# # Split the data into training and testing sets, keeping the same random state for reproducibility
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Display the shapes of the resulting datasets
# print(f"\nShapes - X_train: {X_train.shape}, X_test: {X_test.shape}, y_train: {y_train.shape}, y_test: {y_test.shape}")

In [26]:
# # Reshape X for fitting to LSTM where each step is a single frame
# X_reshaped = X.reshape(X.shape[0], 1, X.shape[1])  # Each row as a single timestep

# # Split the data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y, test_size=0.2, random_state=42)


In [27]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Conv1D, MaxPooling1D, Dense, Dropout, Flatten, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the dataset


# Replace non-drinking dV with 0 and keep drinking ones
df['dV'] = df.apply(lambda row: 0 if row['Label'] == 0 or pd.isna(row['dV']) else row['dV'], axis=1)

# Aggregate data by sip_id, summing the dV for frames of each sip
df_grouped = df.groupby('sip_id').agg({
    **{col: 'mean' for col in df.columns if col.startswith("Zone_")},  # Mean values for zones
    'dV': 'sum',                                      # Total volume for each sip
    'Participant_ID': 'first'  # Retrieve first participant ID for grouping
}).reset_index()

# Prepare features and target variable
sensor_cols = [col for col in df_grouped.columns if col.startswith("Zone_")]
X = df_grouped[sensor_cols + ['Participant_ID']].values
y = df_grouped['dV'].values  # The total volume for each sip

# Reshape X to have each sip's frames as one sequence
X_reshaped = X.reshape(X.shape[0], 1, X.shape[1])  # Each sip as a single timestep

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y, test_size=0.2, random_state=42)


In [13]:
X_train

array([[[1.82144042e+03, 9.92589574e+02, 1.07328444e+03, 9.05665081e+02,
         1.02537112e+03, 1.11634030e+03, 1.62762288e+03, 1.90860695e+03,
         1.57935395e+03, 1.23025383e+03, 9.87497931e+02, 1.15094601e+03,
         1.16551076e+03, 1.53465598e+03, 1.74742780e+03, 1.91927679e+03,
         1.18846938e+03, 1.06194208e+03, 9.19478693e+02, 1.22149152e+03,
         1.49254055e+03, 1.59200414e+03, 1.89850786e+03, 2.03992553e+03,
         8.95235209e+02, 8.31361605e+02, 8.32797269e+02, 1.29907158e+03,
         1.58523066e+03, 1.67521390e+03, 1.76347600e+03, 1.99158378e+03,
         9.32014481e+02, 7.98676045e+02, 7.32050269e+02, 1.03432623e+03,
         1.34266405e+03, 1.32417998e+03, 1.54095428e+03, 1.69373562e+03,
         1.21316115e+03, 8.90124741e+02, 6.32704799e+02, 7.49238933e+02,
         9.85017998e+02, 1.06137319e+03, 1.27125734e+03, 1.86246856e+03,
         1.26404096e+03, 1.01026562e+03, 5.36925941e+02, 5.44020480e+02,
         6.81027927e+02, 8.04742036e+02, 1.10462640

In [14]:
X_test

array([[[1.48666366e+03, 1.51197389e+03, 1.56142617e+03, 1.61300808e+03,
         1.56406683e+03, 1.53877184e+03, 1.76651445e+03, 1.83186665e+03,
         1.69889960e+03, 1.74336058e+03, 1.66589338e+03, 1.74976904e+03,
         1.70040814e+03, 1.75212807e+03, 1.86795586e+03, 1.84271557e+03,
         1.55140224e+03, 1.75031831e+03, 1.74474790e+03, 1.76942213e+03,
         1.83593503e+03, 1.86452191e+03, 1.87900964e+03, 1.83572210e+03,
         1.57282624e+03, 1.70445601e+03, 1.71490208e+03, 1.80939913e+03,
         1.81948057e+03, 1.78224650e+03, 1.78120112e+03, 1.82624806e+03,
         1.47047964e+03, 1.64854834e+03, 1.55020361e+03, 1.53501772e+03,
         1.57148150e+03, 1.51977992e+03, 1.66098166e+03, 1.77837644e+03,
         1.44607554e+03, 1.45433230e+03, 1.06866366e+03, 8.70422443e+02,
         1.01608579e+03, 1.17294902e+03, 1.46360864e+03, 1.74042648e+03,
         1.39770594e+03, 1.16886229e+03, 5.77851725e+02, 4.23633820e+02,
         4.79215729e+02, 8.22317998e+02, 1.25598850

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

data = df[df['Label'] == 1]
data = data.drop(columns=["Volume"])

# Check for missing values and handle them if necessary
print(data.isnull().sum())
print(data)

# Input features: TOF readings from Zone_0 to Zone_63 and some categorical variables
X = data.drop(columns=["dV",'next','prev','Straw','Actual_Volume','sip_start','sip_end' ])
print('x',X)
print('y', y)
y = data["dV"]  # Target variable
print(len(X))  # Length of features
print(len(y))  
# Normalize the input features
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

# Encode categorical variables if necessary
# Use pd.get_dummies if you decide to include Participant_ID or sip_id as features
# Example: X = pd.get_dummies(data, columns=["Participant_ID", "sip_id"])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = LinearRegression()


# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluation metrics
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# Print out the evaluation metrics
print(f'Mean Squared Error (MSE): {mse}')
print(f'Mean Absolute Error (MAE): {mae}')
print(f'Root Mean Squared Error (RMSE): {rmse}')
print(f'R² Score: {r2}')

# Create a comparison table of actual vs predicted values
comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
print(comparison_df)

# Optionally save the comparison dataframe to a CSV file
comparison_df.to_csv('comparison_actual_vs_predicted.csv', index=False)

# Visualization of the predictions vs actual values
plt.scatter(y_test, y_pred)
plt.xlabel('Actual dV Values')
plt.ylabel('Predicted dV Values')
plt.title('Actual vs. Predicted dV Values')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--')  # Diagonal line for reference
plt.show()

Time              0
Zone_0            0
Zone_1            0
Zone_2            0
Zone_3            0
                 ..
sip_start         0
sip_end           0
sip_id            0
dV                0
TOTAL_SIP_TIME    0
Length: 80, dtype: int64
                Time  Zone_0  Zone_1  Zone_2  Zone_3  Zone_4  Zone_5  Zone_6  \
212    1750614894000     259     264     266     264     260     257     241   
213    1750614894200     259     264     266     264     260     257     241   
214    1750614894400     276     272     270     266     260     257     241   
215    1750614894600     277     285    2054     285     283     276     252   
216    1750614894800     276     288    2054     298     286     284     266   
...              ...     ...     ...     ...     ...     ...     ...     ...   
36185  1750885893400    1412     291     283    2496    2597    2660    2667   
36186  1750885893600    1406     287     273    2517    2610    2656    2660   
36187  1750885893800    1366     25

ValueError: could not convert string to float: 'Male'

In [ ]:
import pandas as pd

# Assuming df is your original DataFrame
data = df[df['Label'] == 1]
data = data.drop(columns=["Volume"])

# Check for missing values and handle them if necessary
print(data.isnull().sum())
print(data)

# Input features: TOF readings from Zone_0 to Zone_63, and some categorical variables
# Note: Ensure you handle 'sip_id' if used in the input features
X = data.drop(columns=["dV", 'next', 'prev', 'Straw', 'Actual_Volume', 'sip_start', 'sip_end'])
y = data["dV"]  # Target variable

print('X shape:', X.shape)
print('y shape:', y.shape)

# Unique SIP Sessions
unique_sip_ids = data['sip_id'].unique()

# Define the split using 80% of SIP sessions for training
split_index = int(len(unique_sip_ids) * 0.8)
train_sip_ids = unique_sip_ids[:split_index]

# Create training and testing datasets
train_data = data[data['sip_id'].isin(train_sip_ids)]
test_data = data[~data['sip_id'].isin(train_sip_ids)]

# Separate features and target variables
X_train = train_data.drop(columns=["dV", 'next', 'prev', 'Straw', 'Actual_Volume', 'sip_start', 'sip_end'])
y_train = train_data["dV"]

X_test = test_data.drop(columns=["dV", 'next', 'prev', 'Straw', 'Actual_Volume', 'sip_start', 'sip_end'])
y_test = test_data["dV"]
print('Training set shape:', X_train, y_train)
print('Testing set shape:', X_test, y_test)
# Print shapes of the resulting training and testing sets
print('Training set shape:', X_train.shape, y_train.shape)
print('Testing set shape:', X_test.shape, y_test.shape)

# Normalize the input features if desired
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.optimizers import Adam




# Reshape data for CNN and LSTM
X_train_reshaped = np.reshape(X_train.values, (X_train.shape[0], X_train.shape[1], 1))  # (samples, features, 1)
X_test_reshaped = np.reshape(X_test.values, (X_test.shape[0], X_test.shape[1], 1))  # (samples, features, 1)

# Define input shape for LSTM and CNN
input_shape = (X_train_reshaped.shape[1], 1)  # (timesteps, features)

def build_dnn_model(input_shape):
    model = Sequential()
    model.add(Input(shape=input_shape))  # Use Input layer
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(1, activation='linear'))  # Regression output
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])
    return model

def build_cnn_model(input_shape):
    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))
    
    model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))
    
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.3))
    
    model.add(Dense(1, activation='linear'))  # Regression output
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])
    return model

def build_lstm_model(input_shape):
    model = Sequential()
    model.add(LSTM(128, input_shape=input_shape, return_sequences=True))
    model.add(Dropout(0.3))
    
    model.add(LSTM(64, return_sequences=False))
    model.add(Dropout(0.3))
    
    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.3))
    
    model.add(Dense(1, activation='linear'))  # Regression output
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])
    return model

if __name__ == "__main__":
    # # Training DNN
    # dnn_model = build_dnn_model(input_shape=(X_train.shape[1],))  # Correct input shape
    # dnn_model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=1)

    # # Training CNN
    # cnn_model = build_cnn_model(input_shape=input_shape)  # Correct input shape
    # cnn_model.fit(X_train_reshaped, y_train, epochs=100, batch_size=32, verbose=1)

    # Training LSTM
    lstm_model = build_lstm_model(input_shape=(X_train_reshaped.shape[1], 1))  # Correct input shape
    lstm_model.fit(X_train_reshaped, y_train, epochs=100, batch_size=32, verbose=1)

    # Predictions and evaluations follow here...

In [ ]:
df

In [ ]:
y_train

In [79]:
y_test

755       60.0
5248      80.0
9071      80.0
6046     100.0
15332     40.0
         ...  
2763     100.0
14535    100.0
6075     100.0
2991      80.0
5876      40.0
Name: dV, Length: 263, dtype: float64

In [40]:
X_test

,Time,Zone_0,Zone_1,Zone_2,Zone_3,Zone_4,Zone_5,Zone_6,Zone_7,Zone_8,...,Zone_62,Zone_63,Label,Participant_ID,Gender,Container_Weight,drink,temp,sip_id,TOTAL_SIP_TIME
755,1750615003200,333,331,2204,2225,327,341,338,1594,316,...,304,286,1,1,Male,12.7,water,n,3,13
5248,1750631992800,1692,1658,1636,1623,1604,1593,1579,1566,1659,...,1479,1475,1,5,Male,10.9,water,h,2,28
9071,1750634828600,1549,1540,1543,1537,1531,1533,1528,1541,1546,...,1561,1563,1,8,Male,359.0,water,h,2,32
6046,1750632490400,1627,1609,1583,1588,1580,1577,1560,1562,1601,...,1494,1487,1,6,Male,10.9,water,h,1,46
15332,1750702523400,1591,1608,1630,1634,1667,1692,1723,1770,1559,...,1602,1620,1,13,Male,7.7,water,n,4,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2763,1750616976600,255,250,248,192,220,197,228,160,249,...,280,2095,1,3,Male,12.7,water,c,1,24
14535,1750702363600,1560,1585,1588,1604,1608,1626,1646,1669,1544,...,1576,1589,1,13,Male,7.7,water,n,1,27
6075,1750632496200,1628,1608,1601,1581,1569,1559,1544,1538,1608,...,1490,1479,1,6,Male,10.9,water,h,1,46
2991,1750617022400,330,305,305,2031,2095,330,330,342,396,...,274,285,1,3,Male,12.7,water,c,2,22


In [41]:
    for model_name, model, X_eval in [
        # 'DNN',dnn_model, X_test), ('CNN', cnn_model, X_test_reshaped), 
        ('LSTM', lstm_model, X_test_reshaped)]:
        print(f"Evaluating {model_name}...")
        y_pred = model.predict(X_eval)

        # Calculate evaluation metrics
        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)
        rmspe = np.sqrt(mse) / y_test.mean()
        # Print evaluation metrics
        print(f'{model_name} Evaluation Metrics:')
        print(f'Mean Squared Error (MSE): {mse}')
        print(f'Mean Absolute Error (MAE): {mae}')
        print(f'Root Mean Squared Error (RMSE): {rmse}')
        print(f'R² Score: {r2}')
        print(f"RMSPE {100*np.mean(rmspe):.2f}%")
        # Comparison DataFrame
        comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred.flatten()})
        print(comparison_df.head(50))

Evaluating LSTM...


ValueError: Invalid dtype: object

In [42]:
y_pred

NameError: name 'y_pred' is not defined

In [43]:
y_test.head(50)

755       60.0
5248      80.0
9071      80.0
6046     100.0
15332     40.0
2990      80.0
3198      60.0
8617     100.0
15759     80.0
1698     100.0
14130     40.0
6458      80.0
3427      40.0
13903     60.0
16010     60.0
9081      80.0
748       60.0
4275      60.0
16217     20.0
5544      60.0
753       60.0
9368      60.0
1331      20.0
6929      60.0
9376      60.0
11555     80.0
7645      80.0
6055     100.0
5549      60.0
1042      40.0
14726     80.0
13548     80.0
3755     100.0
9661     100.0
9941      80.0
6456      80.0
15324     40.0
14721     80.0
15140     60.0
10957    100.0
6074     100.0
6436      80.0
6433      80.0
13887     60.0
14129     40.0
14133     40.0
6449      80.0
5550      60.0
6954      60.0
10950    100.0
Name: dV, dtype: float64